In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# 데이터 로드
df_clean = pd.read_csv('/content/drive/MyDrive/취업 포폴/ashrae 에너지 예측 분석/data collection/df_preprocessed.csv')


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

sns.set_style('whitegrid')

In [4]:
def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and 'datetime' not in str(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

df_clean = reduce_memory(df_clean)
print(df_clean.memory_usage(deep=True).sum() / 1024**2, 'MB')

4666.726652145386 MB


In [5]:
df_clean['primary_use'] = df_clean['primary_use'].astype('category')

In [6]:
df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
print(df_clean['timestamp'].dtype)  # datetime64[ns] 나와야 정상

datetime64[ns]


In [7]:
#  사전 준비 - train/valid 분리
df_clean = df_clean.sort_values(['building_id', 'meter', 'timestamp']).reset_index(drop=True)

train_mask = df_clean['timestamp'] < '2016-11-01'
valid_mask = df_clean['timestamp'] >= '2016-11-01'

In [8]:
# 1. 시간 feature
df_clean['hour'] = df_clean['timestamp'].dt.hour
df_clean['day'] = df_clean['timestamp'].dt.day
df_clean['weekday'] = df_clean['timestamp'].dt.dayofweek
df_clean['month'] = df_clean['timestamp'].dt.month
df_clean['is_weekend'] = (df_clean['weekday'] >= 5).astype(int)

In [9]:
# 2. Cyclical encoding
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

df_clean['month_sin'] = np.sin(2 * np.pi * df_clean['month'] / 12)
df_clean['month_cos'] = np.cos(2 * np.pi * df_clean['month'] / 12)

# weekday도 순환 구조라 추가 추천 (월요일-일요일 경계도 이어져야 하니까)
df_clean['weekday_sin'] = np.sin(2 * np.pi * df_clean['weekday'] / 7)
df_clean['weekday_cos'] = np.cos(2 * np.pi * df_clean['weekday'] / 7)

In [10]:
# 3. Lag feature
for lag in [1, 24, 168]:
    df_clean[f'lag_{lag}'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading'].shift(lag)

In [11]:
# 4. Rolling statistics
grouped = df_clean.groupby(['building_id', 'meter'])['log_meter_reading']

df_clean['rolling_mean_24'] = grouped.shift(1).groupby([df_clean['building_id'], df_clean['meter']]).rolling(24).mean().reset_index(level=[0,1], drop=True)

# 1단계: shift(1)부터 별도 컬럼으로 생성 (인덱스 원본 그대로 유지됨)
df_clean['log_meter_reading_shift1'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading'].shift(1)

# 2단계: 그 shift된 값 기준으로 rolling (이것도 인덱스 그대로 유지되니 reset_index 불필요)
df_clean['rolling_mean_24'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading_shift1'].transform(lambda x: x.rolling(24).mean())
df_clean['rolling_std_24'] = df_clean.groupby(['building_id', 'meter'])['log_meter_reading_shift1'].transform(lambda x: x.rolling(24).std())

# 중간 컬럼 정리 (필요 없으면 삭제)
df_clean = df_clean.drop(columns=['log_meter_reading_shift1'])

In [ ]:
# 5. Building feature
df_clean['log_square_feet'] = np.log1p(df_clean['square_feet'])

le_primary_use = LabelEncoder()
df_clean['primary_use_encoded'] = le_primary_use.fit_transform(df_clean['primary_use'])

# year_built은 이미 building_age로 파생해뒀으니 그대로 사용
# building stats - train 기준으로만 계산 (leakage 방지, 지난번 얘기했던 부분)
building_stats = (
    df_clean.loc[train_mask]
    .groupby(['building_id', 'meter'])['log_meter_reading']
    .agg(['mean', 'std']).add_prefix('building_')
    .reset_index()
)
df_clean = df_clean.merge(building_stats, on=['building_id', 'meter'], how='left')

# valid에만 있고 train에 없는 building_id/meter 조합은 NaN 발생 가능 → 전체 평균으로 대체
df_clean['building_mean'] = df_clean['building_mean'].fillna(df_clean.loc[train_mask, 'log_meter_reading'].mean())
df_clean['building_std'] = df_clean['building_std'].fillna(df_clean.loc[train_mask, 'log_meter_reading'].std())

In [ ]:
# 6.CDD/HDD
base_temp = 18.3  # 섭씨 기준 (화씨면 65)

df_clean['CDD'] = (df_clean['air_temperature'] - base_temp).clip(lower=0)
df_clean['HDD'] = (base_temp - df_clean['air_temperature']).clip(lower=0)

In [ ]:
# 7. Weather interaction

# temp x humidity (dew_temperature를 습도 proxy로 사용)
df_clean['temp_x_dew'] = df_clean['air_temperature'] * df_clean['dew_temperature']

# temp x building type - 지난번 얘기했듯 곱셈보다 primary_use별 평균 온도 반응을 보는 게 나을 수 있음
# 대신 group별 평균으로 반영 (LightGBM이 카테고리+수치 조합을 알아서 분리하긴 하지만, 명시적으로 넣고 싶다면)
df_clean['temp_x_primary_use'] = df_clean['air_temperature'] * df_clean['primary_use_encoded']

In [ ]:
# 최종 확인

# lag/rolling으로 생긴 결측치는 자연스러운 것 (앞부분 168시간 이내 데이터)
new_feature_cols = ['hour_sin','hour_cos','month_sin','month_cos','weekday_sin','weekday_cos',
                     'lag_1','lag_24','lag_168','rolling_mean_24','rolling_std_24',
                     'log_square_feet','primary_use_encoded','building_mean','building_std',
                     'CDD','HDD','temp_x_dew','temp_x_primary_use']

print(df_clean[new_feature_cols].isnull().sum())
print(df_clean.shape)